## Table of Contents
### 1. [Import Libraries](#import-libraries)

In [1]:
import os
import zipfile
# Packages for data manipulation
import pandas as pd
import numpy as np

# Packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Packages for machine learning
import sklearn as sk

# Packages for data preprocessing
from sklearn.preprocessing import  RobustScaler, LabelEncoder

# Data splitting, model training, evaluation
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ML Algorithms
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor, SGDClassifier

# Data Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

# Optimization - Hyperparameter tuning
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Optimization - Feature selection
from sklearn.feature_selection import SelectKBest, mutual_info_regression, chi2

random_state = 42

### 2. [Load & Inspect Data](#load--inspect-data)

In [2]:
# Load the original dataset with 'SalePrice' column for target variable and prediction
df_original = pd.read_csv('./data_from_kaggle/train.csv')
# Load the dataset
df = pd.read_csv('./cleaned_data/test_defaultV2.csv')

### 3. [Modeling / Statistical Analysis](#modeling--statistical-analysis) 

   - [Data train/test split](#Data-train-/-test-split)

In [46]:
random_state = 42
x = df
y = df_original['SalePrice']

In [47]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=random_state)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1314, 52), (146, 52), (1314,), (146,))

   - [XGBoost](#XGBoost)

In [48]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [49]:
xgb = XGBRegressor(
    n_estimators=500,      
    learning_rate=0.05,    
    max_depth=6,          
    subsample=0.8,         
    colsample_bytree=0.8,  
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"R²:   {r2:.4f}")
print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")




R²:   0.9277
MAE:  14741.52
MSE:  660299328.00
RMSE: 25696.29


In [50]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42
)

param_dist = {
    "n_estimators": randint(360, 1200),
    "learning_rate": uniform(0.01, 0.2),   # 0.01 ~ 0.21
    "max_depth": randint(3, 15),
    "min_child_weight": randint(1,8),
    "subsample": uniform(0.6, 0.4),        # 0.6 ~ 1.0
    "colsample_bytree": uniform(0.6, 0.4), # 0.6 ~ 1.0
    "gamma": uniform(0.0, 0.5),
    "reg_alpha": uniform(0.0, 0.1),        # L1
    "reg_lambda": uniform(0.7, 0.6)        # L2 (0.7 ~ 1.3)
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,            
    scoring="neg_root_mean_squared_error",
    cv=4,
    random_state=80,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best Params:", search.best_params_)

y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"[Tuned XGB] R²: {r2:.4f} | MAE: {mae:.2f} | MSE: {mse:.2f} | RMSE: {rmse:.2f}")



Fitting 4 folds for each of 50 candidates, totalling 200 fits
Best Params: {'colsample_bytree': 0.6505404250816037, 'gamma': 0.1711063515812276, 'learning_rate': 0.0258036032396442, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 862, 'reg_alpha': 0.0016966687010190108, 'reg_lambda': 1.1633016575424624, 'subsample': 0.7571133679815375}
[Tuned XGB] R²: 0.9281 | MAE: 14740.99 | MSE: 657160768.00 | RMSE: 25635.15


* Train/Test Split : 60/40

[Tuned XGB] R²: 0.8938 | MAE: 15803.81 | MSE: 769711936.00 | RMSE: 27743.68

[Upsampled]R²: 0.8954 | MAE: 15937.49 | MSE: 758203520.00 | RMSE: 27535.50

* Train/Test Split : 70/30

[Tuned XGB]R²: 0.9150 | MAE: 15824.34 | MSE: 593071040.00 | RMSE: 24353.05

[Upsampled]R²: 0.9193 | MAE: 15539.71 | MSE: 563411264.00 | RMSE: 23736.29

* Train/Test Split : 80/20

[Tuned XGB]R²: 0.9233 | MAE: 15418.41 | MSE: 588655488.00 | RMSE: 24262.22

[Upsampled]R²: 0.9171 | MAE: 16160.36 | MSE: 636250112.00 | RMSE: 25224.00

* Train/Test Split : 90/10

[Tuned XGB]R²: 0.9281 | MAE: 14740.99 | MSE: 657160768.00 | RMSE: 25635.15

[Upsampled]R²: 0.9176 | MAE: 15335.15 | MSE: 753197824.00 | RMSE: 27444.45

   - [XGBoost-up-sampling](#XGBoost-up-sampling)

In [51]:
import pandas as pd
from typing import Tuple, Literal, Optional

def upsample_by_bins(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    q: int = 10,
    target_mode: Literal["max", "quantile"] = "max",
    target_quantile: float = 0.8,
    random_state: int = 42,
    shuffle: bool = True,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    """
    依 y 的分位數分箱後對每個箱做上採樣（with replacement）。

    Parameters
    ----------
    X_train : DataFrame
        訓練特徵
    y_train : Series
        訓練目標（連續值，例如 SalePrice）
    q : int, default=10
        qcut 的分箱數（建議 8~12 之間）
    target_mode : {"max", "quantile"}, default="max"
        - "max": 每個箱補到最多樣本的箱數量
        - "quantile": 每個箱補到箱數量的某分位數（由 target_quantile 指定）
    target_quantile : float, default=0.8
        當 target_mode="quantile" 時使用，例如補到箱數量的 80 百分位
    random_state : int
        抽樣亂數種子
    shuffle : bool, default=True
        是否對合併後的資料打散

    Returns
    -------
    X_up : DataFrame
        上採樣後的訓練特徵
    y_up : Series
        上採樣後的訓練目標
    bin_counts_before : Series
        上採樣前各箱樣本數
    bin_counts_after : Series
        上採樣後各箱樣本數
    """
    df = X_train.copy()
    df["_y_"] = y_train.values

    # 依分位數分箱（duplicates="drop" 可避免邊界重疊錯誤）
    df["_bin_"] = pd.qcut(df["_y_"], q=q, labels=False, duplicates="drop")

    bin_counts_before = df["_bin_"].value_counts().sort_index()

    # 決定補到的目標數量
    if target_mode == "max":
        target_n = int(bin_counts_before.max())
    elif target_mode == "quantile":
        target_n = int(bin_counts_before.quantile(target_quantile))
        target_n = max(target_n, int(bin_counts_before.median()))  # 基本保護，避免太小
    else:
        raise ValueError('target_mode must be "max" or "quantile".')

    parts = []
    for b, grp in df.groupby("_bin_"):
        if len(grp) < target_n:
            need = target_n - len(grp)
            up = grp.sample(n=need, replace=True, random_state=random_state)
            grp = pd.concat([grp, up], axis=0)
        parts.append(grp)

    df_up = pd.concat(parts, axis=0)
    if shuffle:
        df_up = df_up.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    bin_counts_after = df_up["_bin_"].value_counts().sort_index()

    X_up = df_up.drop(columns=["_y_", "_bin_"])
    y_up = df_up["_y_"]

    return X_up, y_up, bin_counts_before, bin_counts_after


In [52]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def print_metrics(y_true, y_pred, prefix=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{prefix}R²: {r2:.4f} | MAE: {mae:.2f} | MSE: {mse:.2f} | RMSE: {rmse:.2f}")



In [53]:
# 1) 先做上採樣（只動訓練集）
X_train_up, y_train_up, before, after = upsample_by_bins(
    X_train, y_train,
    q=10,                 # 分箱數（可試 8/10/12）
    target_mode="max",    # 或 "quantile"
    target_quantile=0.8,  # 若 target_mode="quantile" 時才用到
    random_state=42
)

print("Bin counts BEFORE:\n", before)
print("Bin counts AFTER:\n",  after)

# 2) 重新訓練模型（先用你調好的最佳參數；或用同一組 baseline）
from xgboost import XGBRegressor

xgb_up = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

xgb_up.fit(X_train_up, y_train_up)
y_pred_up = xgb_up.predict(X_test)

print_metrics(y_test, y_pred_up, prefix="[Upsampled]")


Bin counts BEFORE:
 _bin_
0    132
1    132
2    131
3    135
4    128
5    130
6    132
7    135
8    127
9    132
Name: count, dtype: int64
Bin counts AFTER:
 _bin_
0    135
1    135
2    135
3    135
4    135
5    135
6    135
7    135
8    135
9    135
Name: count, dtype: int64
[Upsampled]R²: 0.9176 | MAE: 15335.15 | MSE: 753197824.00 | RMSE: 27444.45
